# DL-VINS evaluation analysis

**Naming conventions** (from `scripts/eval_runner.py`):

- Results dir: `tmp/{dataset}_eval_{mode}[_loop]` — `dataset ∈ {euroc, ntu_viral, botanic_garden, subt_mrs}`, `mode ∈ {mono, stereo}`, `_loop` suffix when loop fusion is on.
- Method dir: `dlvins_{ext}_{mode}[_loop]`, `vins_fusion_{mode}[_loop]`, `okvis2_{mode}_{slam|vio}`.
- Per `{method}/{seq}/`: `vio_trajectory_run01.tum` (primary = loop-corrected when `_loop`), `vio_trajectory_raw_run01.tum` (raw pre-loop odometry).
- GT: `{gt_dir}/{seq}_gt.tum`.

**Divergence rule**: a run is flagged **divergent (failure)** when its ATE exceeds
`DIVERGENCE_PCT`% of the GT path length over the associated overlap (default 5% —
e.g. >5 m on a 100 m trajectory). Divergent bars are **crossed out** in the bar chart.

In [ ]:
# check dependencies
import os, copy, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from evo.tools import file_interface
from evo.core import sync, metrics
from evo.core.trajectory import PoseTrajectory3D

# !pip install --upgrade pandas matplotlib evo

In [ ]:
logging.getLogger("evo").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REPO     = Path("/home/ubuntu/dl-vins-factory")
TMP      = REPO / "tmp"
# TMP      = REPO / "results/x86_64/ntu_viral"
# TMP      = REPO / "results/x86_64/botanic_garden"
# TMP      = REPO / "results/x86_64/botanic_garden_rgb"
# TMP      = REPO / "results/jetson_aarch/botanic_garden"
# TMP      = REPO / "results/jetson_aarch/botanic_garden_rgb"
RUN_TAG  = "run01"
MAX_DIFF = 0.05          # GT<->est association window [s]
MIN_POSES = 5            # skip pairs that associate fewer than this
DIVERGENCE_PCT = 5.0    # ATE > this % of GT path length -> divergent (failure)

# dataset -> GT dir (mirrors DATASETS[...]['gt_dir'] in scripts/eval_runner.py)
GT_DIR = {
    "uosm_campus":    REPO / "dataset/1_UoSM-Campus/groundtruth",
    "euroc":          REPO / "dataset/2_EuRoC/groundtruth",
    "kaist_vio":      REPO / "dataset/3_Kaist-VIO/groundtruth",
    "uzh_fpv":        REPO / "dataset/4_UZH-FPV/groundtruth",
    "tum_vi":         REPO / "dataset/5_TUM-VI/groundtruth",
    "subt_mrs":       REPO / "dataset/6_SubT-MRS/groundtruth",
    "ntu_viral":      REPO / "dataset/7_NTU-VIRAL/groundtruth",
    "botanic_garden": REPO / "dataset/8_Botanic-Garden/groundtruth",
    "botanic_garden_rgb": REPO / "dataset/8_Botanic-Garden/groundtruth",
    "aqualoc":        REPO / "dataset/9_Aqualoc/groundtruth",
    "uma_vi":         REPO / "dataset/10_UMA-VI/groundtruth",
}

# NTU VIRAL ground truth tracks the Leica prism, which sits at a fixed lever arm from
# the IMU/body frame. See github.com/ntu-aris/ntu_viral_dataset/issues/20.
NTU_PRISM_OFFSET = np.array([-0.05, 0.0, 0.055])    # prism position in body/IMU frame [m]
GT_BODY_OFFSET   = {"ntu_viral": NTU_PRISM_OFFSET}  # dataset -> est->GT body lever arm

# base-method -> colour
COLORS = {
    "vins_fusion": "#FF0000", "okvis2": "#1f9e89",
    "dlvins_gftt_cpu": "#0062ff",
    "dlvins_sift_cpu_lk": "#6c8091", "dlvins_sift_cpu_lightglue": "#4e4e4e", 
    "dlvins_aliked_lk": "#ff9a60", "dlvins_aliked_lightglue": "#ff4400",
    "dlvins_superpoint_lk": "#7bff9a",  "dlvins_superpoint_lightglue": "#00ca1b",
    "dlvins_raco_lk": "#ff7ad1", "dlvins_raco_lightglue": "#ff00d4", 
    "dlvins_xfeat_lk": "#4ba9a4", "dlvins_xfeat_lightglue": "#0082ae",
}
def color_for(m): return COLORS.get(m, "#705555FF")

In [ ]:
def parse_eval_dir(name):
    """'{dataset}_eval_{mode}[_loop][_tag]' -> (dataset, mode, is_loop) or None.
    Tolerant of a trailing EVAL_RUN_TAG (e.g. 'mono_loop_cmp16')."""
    if "_eval_" not in name:
        return None
    ds, rest = name.split("_eval_", 1)
    if ds not in GT_DIR:
        return None
    is_loop = "_loop" in rest
    mode = rest.split("_loop", 1)[0] if is_loop else rest
    return ds, mode, is_loop

def classify_method(label, mode):
    """-> (base_method, fixed_variant or None). okvis2 fixes loop/noloop via slam/vio."""
    if label.startswith("okvis2_"):
        return "okvis2", ("loop" if label.endswith("_slam") else "noloop")
    for suf in (f"_{mode}_loop", f"_{mode}"):
        if label.endswith(suf):
            return label[: -len(suf)], None
    return label, None

def gt_path(ds, seq):
    return os.path.join(GT_DIR[ds], f"{seq}_gt.tum")

def apply_body_offset(e, offset):
    """Copy of trajectory e with positions moved to a rigidly-attached body-frame
    point (leverarm): P' = T_world_body @ [offset, 1]. Orientation unchanged."""
    off_h = np.array([offset[0], offset[1], offset[2], 1.0])
    new_pos = np.array([(T @ off_h)[:3] for T in e.poses_se3])
    return PoseTrajectory3D(new_pos, e.orientations_quat_wxyz, e.timestamps)

def compute_ate(gt, est, correct_scale=False, max_diff=MAX_DIFF, offset=None):
    """SE3-aligned translation-ATE RMSE.
    `offset` (3-vec, body frame) applies a per-pose leverarm to the estimate before
    alignment (e.g. NTU prism). Returns (rmse, n_assoc, aligned_est, ref_trim, glen)."""
    try:
        ref = file_interface.read_tum_trajectory_file(gt)
        e   = file_interface.read_tum_trajectory_file(est)
        if offset is not None:
            e = apply_body_offset(e, offset)
        ref, e = sync.associate_trajectories(ref, e, max_diff=max_diff)
        if ref.num_poses < MIN_POSES:
            return np.nan, ref.num_poses, None, None, 0.0
        ea = copy.deepcopy(e)
        ea.align(ref, correct_scale=correct_scale)
        ape = metrics.APE(metrics.PoseRelation.translation_part)
        ape.process_data((ref, ea))
        glen = float(getattr(ref, "path_length", 0.0))
        return ape.get_statistic(metrics.StatisticsType.rmse), ref.num_poses, ea, ref, glen
    except Exception as ex:
        print(f"  [warn] ATE failed for {os.path.basename(est)}: {type(ex).__name__}: {ex}")
        return np.nan, 0, None, None, 0.0

In [ ]:
# ---- Discover trajectories under tmp/ and compute ATE (long form) ----
records = []
for d in sorted(TMP.glob("*_eval_*")):
    parsed = parse_eval_dir(d.name)
    if not d.is_dir() or parsed is None:
        continue
    ds, mode, dir_is_loop = parsed
    off = GT_BODY_OFFSET.get(ds)          # body-frame leverarm (e.g. NTU prism); None elsewhere
    for mdir in sorted(p for p in d.iterdir() if p.is_dir()):
        base, fixed_variant = classify_method(mdir.name, mode)
        for sdir in sorted(p for p in mdir.iterdir() if p.is_dir()):
            seq = sdir.name
            gt = gt_path(ds, seq)
            if not os.path.exists(gt):
                continue
            traj = sdir / f"vio_trajectory_{RUN_TAG}.tum"
            raw  = sdir / f"vio_trajectory_raw_{RUN_TAG}.tum"
            entries = []                      # (variant, path)
            if fixed_variant is not None:     # okvis2: dir fixes the variant
                if traj.exists(): entries.append((fixed_variant, traj))
            elif dir_is_loop:                 # dlvins / vins in a _loop run
                if traj.exists(): entries.append(("loop", traj))
                if raw.exists():  entries.append(("noloop", raw))
            else:                             # non-loop run -> primary is raw odom
                if traj.exists(): entries.append(("noloop", traj))
            for variant, path in entries:
                rmse, n, _, _, glen = compute_ate(gt, str(path), offset=off)
                pct = 100.0 * rmse / glen if (np.isfinite(rmse) and glen > 0) else np.nan
                divergent = not np.isfinite(rmse) or bool(np.isfinite(pct) and pct > DIVERGENCE_PCT)
                records.append(dict(dataset=ds, mode=mode, seq=seq, method=base,
                                    variant=variant, ate_rmse=rmse, traj_len_m=glen,
                                    ate_pct=pct, divergent=divergent, n_poses=n,
                                    traj=str(path)))

df = pd.DataFrame.from_records(records)
n_div = int(df["divergent"].sum()) if len(df) else 0
print(f"{len(df)} (method, seq, variant) trajectories across "
      f"{df['dataset'].nunique() if len(df) else 0} datasets | "
      f"{n_div} divergent (ATE > {DIVERGENCE_PCT:.0f}% of GT path length)")
df.sort_values(["dataset", "seq", "method", "variant"]).reset_index(drop=True)

## Loop vs non-loop ATE (all methods)

In [ ]:
if df.empty:
    print("No trajectories found under tmp/.")
    pivot = pd.DataFrame()
else:
    pivot = (df.pivot_table(index=["dataset", "mode", "seq", "method"],
                            columns="variant", values="ate_rmse")
               .reset_index())
    for c in ("noloop", "loop"):
        if c not in pivot.columns:
            pivot[c] = np.nan
    pivot = pivot.rename(columns={"loop": "ATE_loop", "noloop": "ATE_noloop"})
    pivot["loop_improve_%"] = 100.0 * (pivot["ATE_noloop"] - pivot["ATE_loop"]) / pivot["ATE_noloop"]
    dmap = {(r.dataset, r.mode, r.seq, r.method, r.variant): r.divergent for r in df.itertuples()}
    def mark(row, var, col):
        v = row[col]
        if np.isfinite(v) and dmap.get((row.dataset, row.mode, row.seq, row.method, var), False):
            return f"{v:,.4f} ‡"   # double-dagger = divergent
        return f"{v:,.4f}" if np.isfinite(v) else ""
    disp = pivot.copy()
    disp["ATE_noloop"] = pivot.apply(lambda r: mark(r, "noloop", "ATE_noloop"), axis=1)
    disp["ATE_loop"]   = pivot.apply(lambda r: mark(r, "loop",   "ATE_loop"),   axis=1)
    pivot = pivot.sort_values(["dataset", "mode", "seq", "method"]).reset_index(drop=True)
    disp = disp.sort_values(["dataset", "mode", "seq", "method"]).reset_index(drop=True)
disp if not df.empty else None

## Overall ATE mean — loop vs non-loop

In [ ]:
# ── Overall ATE mean across the dataset, split loop vs non-loop ──
# Aggregates the long-form `df` (per method/seq/variant) into mean ATE per
# (dataset, mode, method). Divergent runs are dropped so one blow-up can't skew the mean.
if df.empty:
    print("No trajectories found under tmp/.")
    ate_mean = pd.DataFrame()
else:
    valid = df[np.isfinite(df.ate_rmse) & ~df.divergent]

    def _mean_table(data, keys):
        m = data.pivot_table(index=keys, columns="variant", values="ate_rmse", aggfunc="mean")
        c = data.pivot_table(index=keys, columns="variant", values="ate_rmse", aggfunc="size")
        for col in ("noloop", "loop"):
            if col not in m.columns: m[col] = np.nan
            if col not in c.columns: c[col] = np.nan
        out = pd.DataFrame(index=m.index)
        out["ATE_noloop"]     = m["noloop"]
        out["ATE_loop"]       = m["loop"]
        out["loop_improve_%"] = 100.0 * (m["noloop"] - m["loop"]) / m["noloop"]
        out["n_noloop"]       = c["noloop"].astype("Int64")
        out["n_loop"]         = c["loop"].astype("Int64")
        return out

    per_ds  = _mean_table(valid, ["dataset", "mode", "method"])
    overall = _mean_table(valid.assign(dataset="(all)"), ["dataset", "mode", "method"])
    ate_mean = pd.concat([per_ds, overall]).sort_index()
    n_drop = int((~np.isfinite(df.ate_rmse) | df.divergent).sum())
    print(f"Mean ATE over {len(valid)} valid runs "
          f"({n_drop} divergent/NaN excluded) — split loop vs non-loop")

ate_mean if not df.empty else None

## Per-method ranking

Ranks methods by **mean ATE over the common set of sequences** 

In [ ]:
# ── Per-method ranking on the common sequence subset ──
RANK_VARIANT = "noloop"   # "noloop" = raw VIO drift (isolates the frontend); "loop" = loop-corrected
MIN_COVERAGE = 0.8        # methods below this fraction of a dataset's seqs are listed but unranked

def rank_one(dd, variant):
    v = dd[(dd.variant == variant) & np.isfinite(dd.ate_rmse) & ~dd.divergent]
    if v.empty:
        return None, 0
    seqs_all = v.seq.unique()
    cov  = v.groupby("method").seq.nunique() / len(seqs_all)
    core = cov[cov >= MIN_COVERAGE].index                      # fully-covered methods
    common = (set.intersection(*[set(v[v.method == m].seq) for m in core])
              if len(core) else set(seqs_all))                 # seqs every core method finished
    g = v[v.seq.isin(common)].groupby("method").ate_rmse
    out = pd.DataFrame({"ATE_mean": g.mean(), "ATE_median": g.median(),
                        "n_common": g.size()}).reindex(v.method.unique())
    out["n_total"]   = v.groupby("method").seq.nunique().reindex(out.index)
    out["coverage%"] = (cov * 100).reindex(out.index)
    out["full"]      = out.index.isin(core)
    out = out.sort_values(["full", "ATE_mean"], ascending=[False, True])
    rank = np.where(out["full"].values, np.cumsum(out["full"].values), pd.NA)   # rank full-cov only
    out.insert(0, "rank", pd.array(rank, dtype="Int64"))
    out = out.drop(columns="full")
    out["n_common"] = out["n_common"].astype("Int64")
    out["n_total"]  = out["n_total"].astype("Int64")
    return out, len(common)

if df.empty:
    print("No trajectories found under tmp/.")
else:
    ranks = {}
    for (ds, mode), dd in df.groupby(["dataset", "mode"]):
        r, ncommon = rank_one(dd, RANK_VARIANT)
        if r is None:
            continue
        ranks[(ds, mode)] = r
        print(f"\n=== {ds} ({mode}) — {RANK_VARIANT} ATE over {ncommon} common seqs ===")
        display(r)
    # Overall winner: average each method's per-dataset rank (scale-free; partial-cov rows dropped).
    if ranks:
        allr = pd.concat([r.assign(dataset=k[0], mode=k[1]) for k, r in ranks.items()])
        overall = (allr.dropna(subset=["rank"]).groupby("method")
                       .agg(mean_rank=("rank", "mean"), datasets=("rank", "size"),
                            mean_ATE=("ATE_mean", "mean"))
                       .sort_values("mean_rank"))
        print(f"\n=== Overall ({RANK_VARIANT}): mean rank across datasets (lower = better) ===")
        display(overall)

## ATE bar chart — horizontal ATE, vertical sequence

One figure per dataset, **linear x-axis**. Bars grouped per sequence (y-axis); each
method×variant is a bar (solid = loop, hatched/faded = non-loop). **Divergent runs are not
drawn** — their slot is left blank and marked `✗ FAIL`.

In [ ]:
def barh_dataset(ds, dd):
    seqs   = sorted(dd["seq"].unique())
    groups = sorted(dd.groupby(["method", "variant"]).groups.keys())
    n      = max(len(groups), 1)
    h      = 0.8 / n
    y      = np.arange(len(seqs))
    fig, ax = plt.subplots(figsize=(11, max(2.4, 0.8 * len(seqs) * n ** 0.5)))
    for i, (method, variant) in enumerate(groups):
        vals, divs = [], []
        for s in seqs:
            r = dd[(dd.seq == s) & (dd.method == method) & (dd.variant == variant)]
            vals.append(r["ate_rmse"].values[0] if len(r) else np.nan)
            divs.append(bool(r["divergent"].values[0]) if len(r) else False)
        off = (i - (n - 1) / 2) * h
        ys = y + off
        # Divergent runs are not drawn; the slot is marked "FAIL" instead.
        plot_vals = [np.nan if dv else v for v, dv in zip(vals, divs)]
        ax.barh(ys, plot_vals, height=h,
                label=f"{method} ({variant})", color=color_for(method),
                alpha=1.0 if variant == "loop" else 0.75,
                hatch="" if variant == "loop" else "////",
                edgecolor="black", linewidth=0.4)
        for v, dv, yc in zip(vals, divs, ys):
            if dv:
                ax.text(0, yc, " ✗ FAIL", va="center", ha="left",
                        fontsize=6, color="red", fontweight="bold")
            elif np.isfinite(v):
                ax.text(v, yc, f" {v:.2f}", va="center", ha="left", fontsize=6)
    ax.set_yticks(y); ax.set_yticklabels(seqs)
    ax.set_xlim(left=0)
    ax.set_xlabel("ATE RMSE [m]"); ax.set_ylabel("sequence")
    ax.set_title(f"{ds} ({dd['mode'].iloc[0]}) — ATE vs GT   "
                 f"(✗ FAIL = divergent: ATE > {DIVERGENCE_PCT:.0f}% of GT path len)")
    ax.grid(axis="x", which="both", alpha=0.3)
    ax.legend(fontsize=7, ncol=2, loc="lower right")
    fig.tight_layout()
    plt.show()

# One figure per (dataset, mode) -> mono and stereo are shown as separate charts
if not df.empty:
    for (ds, mode), dd in df.groupby(["dataset", "mode"]):
        barh_dataset(ds, dd)

## Trajectory plots — per sequence, per dataset

Top-down (x–y) overlay of GT (black) vs each method's trajectory, SE3-aligned to GT.
Uses the loop-corrected trajectory when available, else the raw/noloop one. **Divergent
runs are omitted** (listed in the subplot title) so they don't blow up the axis scale.

In [ ]:
import textwrap

def plot_traj_ax(ds, mode, seq, dd, ax):
    gt = gt_path(ds, seq)
    safe_seq = seq.replace('_', r'\_')
    off = GT_BODY_OFFSET.get(ds)          # body-frame leverarm (NTU prism); None elsewhere
    try:
        ref_full = file_interface.read_tum_trajectory_file(gt)
        ax.plot(ref_full.positions_xyz[:, 0], ref_full.positions_xyz[:, 1],
                color="k", lw=1.6, label="GT", zorder=1)
    except Exception:
        ax.set_title(f"{safe_seq} (GT read err)"); return
    omitted = []
    for method in sorted(dd.method.unique()):
        sub = dd[dd.method == method]
        row = sub[sub.variant == "loop"]
        if row.empty:
            row = sub[sub.variant == "noloop"]
        if row.empty:
            continue
        if bool(row.divergent.values[0]):     # skip divergent runs entirely
            omitted.append(method)
            continue
        rmse, n, ea, _, _ = compute_ate(gt, row.traj.values[0], offset=off)
        if ea is None:
            continue
        ax.plot(ea.positions_xyz[:, 0], ea.positions_xyz[:, 1],
                color=color_for(method), lw=1.1, alpha=0.9,
                label=f"{method} [{row.variant.values[0]}] {rmse:.2f}m")
    ax.set_aspect("equal", adjustable="datalim")
    if omitted:
        omitted_text = f"Failed: {', '.join(omitted)}"
        wrapped_text = textwrap.fill(omitted_text, width=55)
        title = f"$\\mathbf{{{safe_seq}}}$\n{wrapped_text}"
    else:
        title = f"$\\mathbf{{{safe_seq}}}$"
        
    ax.set_title(title, fontsize=9, color="black")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.grid(alpha=0.3); ax.legend(fontsize=6, loc="best")

# One figure per (dataset, mode) -> separate mono and stereo trajectory grids.
if not df.empty:
    for (ds, mode), dd_ds in df.groupby(["dataset", "mode"]):
        seqs = sorted(dd_ds["seq"].unique())
        ncols = min(3, len(seqs)); nrows = int(np.ceil(len(seqs) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows),
                                 squeeze=False)
        for k, seq in enumerate(seqs):
            plot_traj_ax(ds, mode, seq, dd_ds[dd_ds.seq == seq], axes[k // ncols][k % ncols])
        for k in range(len(seqs), nrows * ncols):
            axes[k // ncols][k % ncols].axis("off")
        fig.suptitle(f"{ds} ({mode}) — trajectories (x–y, SE3-aligned to GT)", y=1.0)
        fig.tight_layout()
        plt.show()

In [ ]:
import textwrap
import numpy as np
import matplotlib.pyplot as plt

def plot_traj_ax(ds, mode, seq, dd, ax):
    gt = gt_path(ds, seq)
    safe_seq = seq.replace('_', r'\_')
    off = GT_BODY_OFFSET.get(ds)          # body-frame leverarm (NTU prism); None elsewhere
    
    try:
        ref_full = file_interface.read_tum_trajectory_file(gt)
        # ADDED Z-AXIS: ref_full.positions_xyz[:, 2]
        ax.plot(ref_full.positions_xyz[:, 0], ref_full.positions_xyz[:, 1], ref_full.positions_xyz[:, 2],
                color="k", lw=1.6, label="GT", zorder=1)
    except Exception:
        ax.set_title(f"{safe_seq} (GT read err)"); return
        
    omitted = []
    for method in sorted(dd.method.unique()):
        sub = dd[dd.method == method]
        row = sub[sub.variant == "loop"]
        if row.empty:
            row = sub[sub.variant == "noloop"]
        if row.empty:
            continue
        if bool(row.divergent.values[0]):     # skip divergent runs entirely
            omitted.append(method)
            continue
            
        rmse, n, ea, _, _ = compute_ate(gt, row.traj.values[0], offset=off)
        if ea is None:
            continue
            
        # ADDED Z-AXIS: ea.positions_xyz[:, 2]
        ax.plot(ea.positions_xyz[:, 0], ea.positions_xyz[:, 1], ea.positions_xyz[:, 2],
                color=color_for(method), lw=1.1, alpha=0.9,
                label=f"{method} [{row.variant.values[0]}] {rmse:.2f}m")
                
    # 3D Equal aspect ratio handling
    try:
        ax.set_aspect("equal", adjustable="datalim")
    except NotImplementedError:
        # Fallback for some Matplotlib versions where 'equal' isn't fully supported in 3D
        ax.set_box_aspect((1, 1, 1)) 
        
    if omitted:
        omitted_text = f"(Tracking failed: {', '.join(omitted)})"
        wrapped_text = textwrap.fill(omitted_text, width=55)
        title = f"$\\mathbf{{{safe_seq}}}$\n{wrapped_text}"
    else:
        title = f"$\\mathbf{{{safe_seq}}}$"
        
    ax.set_title(title, fontsize=9, color="black")
    
    # ADDED Z-LABEL
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
    ax.grid(alpha=0.3); ax.legend(fontsize=6, loc="best")

# One figure per (dataset, mode) -> separate mono and stereo trajectory grids.
if not df.empty:
    for (ds, mode), dd_ds in df.groupby(["dataset", "mode"]):
        seqs = sorted(dd_ds["seq"].unique())
        ncols = min(3, len(seqs)); nrows = int(np.ceil(len(seqs) / ncols))
        
        fig, axes = plt.subplots(nrows, ncols, figsize=(6.0 * ncols, 5.0 * nrows),
                                 subplot_kw={'projection': '3d'}, squeeze=False)
                                 
        for k, seq in enumerate(seqs):
            plot_traj_ax(ds, mode, seq, dd_ds[dd_ds.seq == seq], axes[k // ncols][k % ncols])
            
        for k in range(len(seqs), nrows * ncols):
            axes[k // ncols][k % ncols].axis("off")
            
        # Updated title to reflect 3D
        fig.suptitle(f"{ds} ({mode}) — trajectories (x–y–z, SE3-aligned to GT)", y=1.0)
        fig.tight_layout()
        plt.show()

## Tracker & Estimator Diagnostics

Four diagnostics isolate **where** each method fails:

1. **Forward tracking success rate** — optical-flow survival per frame bin  
2. **Stereo success rate** — stereo-match acceptance per frame bin  
3. **Stereo acceptance (table, LightGlue path only)**
4. **Backend solver cost** — `final_cost` over the sequence (log scale, binned)

In [ ]:
# ── Load tracker + estimator CSVs from results/ ──────────────────────────────
RESULTS_DIR = REPO / "tmp" / "botanic_garden_eval_stereo_loop"
# RESULTS_DIR = REPO / "results" / "x86_64" / "euroc" / "euroc_eval_stereo_loop"
# RESULTS_DIR = REPO / "results" / "x86_64" / "subt_mrs" / "subt_mrs_eval_nibi_loop"
# RESULTS_DIR = REPO / "results" / "jetson_aarch" / "ntu_viral" / "ntu_viral_eval_stereo_loop"

_MODE_SUFFIXES = ("_stereo_loop", "_mono_loop", "_stereo", "_mono", "_loop")

def _base(name):
    for s in _MODE_SUFFIXES:
        if name.endswith(s):
            return name[: -len(s)]
    return name

def _load_csvs(results_dir, template):
    """Returns {method_base: DataFrame} with 'seq' and 'frame_pct' columns."""
    out = {}
    for mdir in sorted(results_dir.iterdir()):
        if not mdir.is_dir():
            continue
        base = _base(mdir.name)
        parts = []
        for sdir in sorted(mdir.iterdir()):
            if not sdir.is_dir():
                continue
            f = sdir / template.format(run=RUN_TAG)
            if not f.exists():
                continue
            d = pd.read_csv(f)
            d["seq"] = sdir.name
            n = len(d)
            d["frame_pct"] = np.linspace(0.0, 1.0, max(n, 2))
            parts.append(d)
        if parts:
            out[base] = pd.concat(parts, ignore_index=True)
    return out

tracker_all  = _load_csvs(RESULTS_DIR, "feature_tracker_{run}.csv")
estimator_all = _load_csvs(RESULTS_DIR, "estimator_metrics_{run}.csv")

# Short display names and consistent order
METHOD_ORDER = [
    "dlvins_gftt_cpu",
    "dlvins_sift_cpu_lightglue",
    "dlvins_sift_cpu_lk",
    "dlvins_aliked_lightglue",
    "dlvins_aliked_lk",
    "dlvins_raco_lightglue",
    "dlvins_raco_lk",
    "dlvins_superpoint_lightglue",
    "dlvins_superpoint_lk",
    "dlvins_xfeat_lightglue",
    "dlvins_xfeat_lk",
]
METHOD_LABELS = {
    "dlvins_gftt_cpu":                "gftt (classical)",
    "dlvins_sift_cpu_lightglue":      "SIFT+LG",
    "dlvins_sift_cpu_lk":             "SIFT+LK",
    "dlvins_aliked_lightglue":        "ALIKED+LG",
    "dlvins_aliked_lk":               "ALIKED+LK",
    "dlvins_raco_lightglue":          "RACO+LG",
    "dlvins_raco_lk":                 "RACO+LK",
    "dlvins_superpoint_lightglue":    "SuperPoint+LG",
    "dlvins_superpoint_lk":           "SuperPoint+LK",
    "dlvins_xfeat_lightglue":         "XFeat+LG",
    "dlvins_xfeat_lk":                "XFeat+LK",
}

print("Loaded methods:")
for m in METHOD_ORDER:
    if m not in tracker_all:
        print(f"  {m}: MISSING"); continue
    v = tracker_all[m]
    print(f"  {METHOD_LABELS[m]}: {len(v):,} frames, "
          f"{v['seq'].nunique()} sequences")


In [ ]:
# ── Shared layout constants for all per-sequence line charts ─────────────────
SMOOTH_W = 10          # rolling-window width for per-frame rate smoothing
NCOLS    = 4
SEQS     = sorted(next(iter(tracker_all.values()))["seq"].unique())
NROWS    = int(np.ceil(len(SEQS) / NCOLS))
n_methods = sum(m in tracker_all for m in METHOD_ORDER)
n_est     = sum(m in estimator_all for m in METHOD_ORDER)

def _seq_legend(fig, methods, data_dict, ncol=None):
    handles = [plt.Line2D([0], [0], color=color_for(m), lw=2)
               for m in methods if m in data_dict]
    labels  = [METHOD_LABELS[m] for m in methods if m in data_dict]
    fig.legend(handles, labels, fontsize=8, ncol=ncol or len(handles),
               loc="lower center", bbox_to_anchor=(0.5, 0))

# ── Plot 1: Forward tracking success rate — per sequence ─────────────────────
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(5 * NCOLS, 3 * NROWS), squeeze=False)
for k, seq in enumerate(SEQS):
    ax = axes[k // NCOLS][k % NCOLS]
    for m in [m for m in METHOD_ORDER if m in tracker_all]:
        sub = tracker_all[m][tracker_all[m]["seq"] == seq]
        if sub.empty: continue
        s = sub[sub["forward_total"] > 0]
        rate = s["forward_success"] / s["forward_total"]
        sm   = rate.rolling(SMOOTH_W, center=True, min_periods=5).mean()
        ax.plot(s["frame_id"].values, sm.values, color=color_for(m), lw=1.5)
    ax.set_title(seq, fontsize=9); ax.set_ylim(0, 1.05)
    ax.set_xlabel("frame"); ax.set_ylabel("rate")
    ax.grid(alpha=0.25)

for k in range(len(SEQS), NROWS * NCOLS):
    axes[k // NCOLS][k % NCOLS].axis("off")
_seq_legend(fig, METHOD_ORDER, tracker_all)
fig.suptitle(f"Forward Tracking Success Rate  (rolling window = {SMOOTH_W} frames)", fontsize=11)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


In [ ]:
# ── Plot 2: Stereo matching success rate — per sequence ──────────────────────
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(5 * NCOLS, 3 * NROWS), squeeze=False)
for k, seq in enumerate(SEQS):
    ax = axes[k // NCOLS][k % NCOLS]
    for m in [m for m in METHOD_ORDER if m in tracker_all]:
        sub = tracker_all[m][tracker_all[m]["seq"] == seq]
        if sub.empty: continue
        s = sub[sub["stereo_candidates"] > 0]
        rate = s["stereo_success"] / s["stereo_candidates"]
        sm   = rate.rolling(SMOOTH_W, center=True, min_periods=5).mean()
        ax.plot(s["frame_id"].values, sm.values, color=color_for(m), lw=1.5)
    ax.set_title(seq, fontsize=9)
    ax.set_xlabel("frame"); ax.set_ylabel("rate")
    ax.grid(alpha=0.25)

for k in range(len(SEQS), NROWS * NCOLS):
    axes[k // NCOLS][k % NCOLS].axis("off")
_seq_legend(fig, METHOD_ORDER, tracker_all)
fig.suptitle(f"Stereo Matching Success Rate  [rolling window = {SMOOTH_W} frames]", fontsize=11)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


In [ ]:
# ── Plot 3 (table): Stereo acceptance + RANSAC removal — LightGlue path only ──
FAILURE_BITS = {1: "NaN", 2: "Vel", 4: "PosJump", 8: "AccelBias", 16: "GyroBias"}

def decode_failure(series):
    """OR the per-frame failure_reason bitmask over a run.
    Returns (label, n_frames_flagged): label is '—' if healthy, else 'NaN|Vel|…'."""
    vals = series.dropna().astype(int)
    if vals.empty:
        return "—", 0
    mask = int(np.bitwise_or.reduce(vals.values))
    n_flagged = int((vals != 0).sum())
    if mask == 0:
        return "—", 0
    names = [name for bit, name in FAILURE_BITS.items() if mask & bit]
    return "|".join(names), n_flagged

LG_METHODS = [m for m in METHOD_ORDER if m.endswith("_lightglue") and m in tracker_all]

def _csum(tsub, col):
    return int(tsub[col].sum()) if col in tsub.columns else 0

rows = []
for m in LG_METHODS:
    tdf = tracker_all[m]
    edf = estimator_all.get(m)
    for seq in sorted(tdf["seq"].unique()):
        tsub = tdf[tdf["seq"] == seq]
        cand = _csum(tsub, "stereo_candidates")
        succ = _csum(tsub, "stereo_success")
        row = {"method": METHOD_LABELS[m], "seq": seq,
               "cand": cand, "admit": succ,
               "admit_%": round(100.0 * succ / cand, 1) if cand else float("nan"),
               "RANSAC_T": _csum(tsub, "ransac_rejected_temporal")}
        if edf is not None and "failure_reason" in edf.columns:
            reason, n_flagged = decode_failure(edf.loc[edf["seq"] == seq, "failure_reason"])
        else:
            reason, n_flagged = "n/a", 0
        row["failure"] = reason       # estimator FailureReason bits that fired this run
        row["fail_frames"] = n_flagged
        rows.append(row)

if rows:
    reject_table = pd.DataFrame(rows).set_index(["method", "seq"])
    n_fail = int((reject_table["failure"].isin(["—", "n/a"]) == False).sum())
    print(f"LightGlue stereo acceptance / RANSAC — {len(LG_METHODS)} methods; "
          f"{n_fail} / {len(reject_table)} runs tripped the estimator failure bitmask")
    print("Per-candidate reject mask removed; _T = temporal-L E-matrix pre-filter")
    with pd.option_context("display.max_rows", None, "display.width", 200):
        display(reject_table)
else:
    print("No LightGlue tracker data under", RESULTS_DIR)


In [ ]:
# ── Plot 4: Backend solver cost (final_cost) — per sequence ──────────────────
#   solid  = LightGlue / classical (gftt)
#   dashed = LK counterpart
def _linestyle(m):
    return "--" if m.endswith("_lk") else "-"

P4_METHODS = [m for m in METHOD_ORDER if m in estimator_all]
fig, axes = plt.subplots(NROWS, NCOLS, figsize=(5 * NCOLS, 3 * NROWS), squeeze=False)
for k, seq in enumerate(SEQS):
    ax = axes[k // NCOLS][k % NCOLS]
    for m in P4_METHODS:
        if m is ("dlvins_sift_cpu_lk"):
            continue
        sub = estimator_all[m][estimator_all[m]["seq"] == seq]
        if sub.empty: continue
        sm = sub["final_cost"].rolling(SMOOTH_W, center=True, min_periods=5).mean()
        ax.plot(sub["frame_id"].values, sm.values, color=color_for(m),
                lw=1.5, ls=_linestyle(m))
    ax.set_title(seq, fontsize=9)
    ax.set_yscale("log")
    ax.set_xlabel("frame"); ax.set_ylabel("cost [log]")
    ax.grid(axis="both", which="both", alpha=0.25)

for k in range(len(SEQS), NROWS * NCOLS):
    axes[k // NCOLS][k % NCOLS].axis("off")

# legend carries both colour (method) and linestyle (solid=matcher/classical, dashed=LK)
handles = [plt.Line2D([0], [0], color=color_for(m), lw=2, ls=_linestyle(m)) for m in P4_METHODS]
labels  = [METHOD_LABELS[m] for m in P4_METHODS]
fig.legend(handles, labels, fontsize=8, ncol=min(len(handles), 6),
           loc="lower center", bbox_to_anchor=(0.5, 0))
fig.suptitle(f"Backend Solver Final Cost  [log scale, rolling window = {SMOOTH_W} frames]", fontsize=11)
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()


In [ ]:
# ── Divergence root-cause summary ─────────────────────────
import warnings
_DIAG_SEQS = ["V2_02_medium"]
_DIAG_SEQS = ["sbs_01", "sbs_02", "sbs_03"]
_DIAG_SEQS = ["1005_07", "1018-00"]
_DIAG_COLS = {
    "tracked_features":      "Tracked feat",
    "mature_features":        "Mature feat (obs≥4)",
    "stereo_anchored_features": "Stereo-anchored",
    "parallax":               "Parallax (px)",
    "final_cost":             "Backend cost (med)",
    "converged":              "Conv. rate",
    "velocity_norm":          "Velocity norm",
    "accel_bias_norm":        "|Ba|",
    "gyro_bias_norm":         "|Bg|",
}
rows = []
for seq in _DIAG_SEQS:
    for m in METHOD_ORDER:
        if m not in estimator_all: continue
        sub = estimator_all[m][estimator_all[m]["seq"] == seq]
        if sub.empty: continue
        row = {"seq": seq, "method": m.replace("dlvins_", "").replace("_stereo_loop", "")}
        for col, _ in _DIAG_COLS.items():
            if col == "converged":
                row[col] = f"{sub[col].mean():.2f}"
            elif col in sub.columns:
                row[col] = f"{sub[col].median():.1f}"
            else:
                row[col] = "N/A"
        rows.append(row)

import pandas as pd
_df = pd.DataFrame(rows).rename(columns=_DIAG_COLS)
with pd.option_context("display.max_columns", None, "display.width", 200, "display.max_rows", 50):
    display(_df.set_index(["seq", "method"]))


## Runtime cost analysis

**Per-frame runtime**, split by **mode** (mono/stereo) and **dataset** (≈ camera input resolution).
The front-end (feature-tracker thread) and back-end (estimator thread) run **concurrently on separate
threads**, so they are reported and plotted **separately — never summed**. The sustained frame rate is
bounded by the *slower* thread: `pipeline_fps = 1000 / max(front-end, back-end)`, not by their sum.

In [ ]:
# ── Runtime cost per (dataset, mode, method) — front-end and back-end kept SEPARATE ──
RUNTIME_WARMUP = 20  # frames dropped per sequence before taking medians (skip init spikes)

def runtime_cost(root):
    """Median per-frame front-end + back-end runtime per (dataset, mode, method) under an eval root."""
    ft_rows, es_rows = [], []
    for d in sorted(Path(root).glob("*_eval_*")):
        parsed = parse_eval_dir(d.name)
        if not d.is_dir() or parsed is None:
            continue
        ds, mode, _ = parsed
        for mdir in sorted(p for p in d.iterdir() if p.is_dir()):
            method = _base(mdir.name)
            for sdir in sorted(p for p in mdir.iterdir() if p.is_dir()):
                ft = sdir / f"feature_tracker_{RUN_TAG}.csv"
                es = sdir / f"estimator_metrics_{RUN_TAG}.csv"
                if ft.exists():
                    ft_rows.append(pd.read_csv(ft).assign(dataset=ds, mode=mode, method=method, seq=sdir.name))
                if es.exists():
                    es_rows.append(pd.read_csv(es).assign(dataset=ds, mode=mode, method=method, seq=sdir.name))
    if not ft_rows:
        return pd.DataFrame()
    key = ["dataset", "mode", "method"]
    # Front-end (feature-tracker thread): total = extraction + matcher + other.
    ft = pd.concat(ft_rows, ignore_index=True)
    ft["_i"] = ft.groupby(key + ["seq"]).cumcount()
    fe = (ft[ft["_i"] >= RUNTIME_WARMUP]
          .groupby(key)[["extraction_time_ms", "matcher_time_ms", "total_time_ms"]].median())
    fe["frontend_other_ms"] = (fe["total_time_ms"] - fe["extraction_time_ms"].fillna(0)
                               - fe["matcher_time_ms"].fillna(0)).clip(lower=0)
    fe["fe_fps"] = 1000.0 / fe["total_time_ms"].replace(0, np.nan)
    # Back-end (estimator thread): solver ⊂ backend_total (+marg) ⊂ estimator_total.
    if es_rows:
        es = pd.concat(es_rows, ignore_index=True)
        es["_i"] = es.groupby(key + ["seq"]).cumcount()
        es = es[es["_i"] >= RUNTIME_WARMUP]
        be_cols = [c for c in ["solver_time_ms", "backend_total_time_ms", "estimator_total_time_ms"]
                   if c in es.columns]
        be = es.groupby(key)[be_cols].median()
    else:
        be = pd.DataFrame()
    cost = fe.join(be, how="left")
    cost["backend_marg_ms"] = (cost.get("backend_total_time_ms", cost.get("estimator_total_time_ms"))
                               - cost.get("solver_time_ms", 0)).clip(lower=0)
    cost["estimator_other_ms"] = (cost.get("estimator_total_time_ms", 0)
                                  - cost.get("backend_total_time_ms", 0)).clip(lower=0)
    cost["be_fps"] = 1000.0 / cost.get("estimator_total_time_ms",
                                       pd.Series(np.nan, index=cost.index)).replace(0, np.nan)
    # Realistic 2-stage pipeline throughput: bounded by the slower thread (NOT the sum).
    cost["bottleneck_ms"] = cost[["total_time_ms", "estimator_total_time_ms"]].max(axis=1)
    cost["pipeline_fps"] = 1000.0 / cost["bottleneck_ms"].replace(0, np.nan)
    return cost

def _plot_stacked(cost, segs, suptitle):
    pairs = sorted(cost.reset_index().groupby(["dataset", "mode"]).groups.keys())
    if not pairs:
        return
    max_methods = int(cost.reset_index().groupby(["dataset", "mode"]).size().max())
    ncols = min(2, len(pairs)); nrows = int(np.ceil(len(pairs) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(8 * ncols, max(2.2, 0.55 * max_methods + 1.4) * nrows),
                             squeeze=False)
    for idx, (ds, mode) in enumerate(pairs):
        ax = axes[idx // ncols][idx % ncols]
        c = cost.loc[(ds, mode)]
        order = [m for m in METHOD_ORDER if m in c.index] + [m for m in c.index if m not in METHOD_ORDER]
        c = c.reindex(order)
        y = np.arange(len(c)); left = np.zeros(len(c))
        for col, lab, color in segs:
            vals = c[col].fillna(0).values if col in c.columns else np.zeros(len(c))
            ax.barh(y, vals, left=left, color=color, label=lab, edgecolor="white", lw=0.4)
            left += vals
        for yi, tot in zip(y, left):
            ax.text(tot, yi, f" {tot:.1f}ms", va="center", fontsize=7)
        ax.set_yticks(y); ax.set_yticklabels([METHOD_LABELS.get(m, m) for m in c.index], fontsize=8)
        ax.set_xlabel("median per-frame time [ms]"); ax.set_title(f"{ds} ({mode})")
        ax.grid(axis="x", alpha=0.3)
    for idx in range(len(pairs), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=col) for _, _, col in segs]
    fig.legend(handles, [lab for _, lab, _ in segs], ncol=len(segs),
               loc="lower center", bbox_to_anchor=(0.5, 0))
    fig.suptitle(suptitle, y=1.0); fig.tight_layout(rect=[0, 0.06, 1, 1]); plt.show()

FE_SEG = [("extraction_time_ms", "extraction (detect+describe)", "#ff7f0e"),
          ("matcher_time_ms",    "matcher (LightGlue)",          "#1f77b4"),
          ("frontend_other_ms",  "front-end other",              "#9467bd")]
BE_SEG = [("solver_time_ms",     "Ceres solve",     "#2ca02c"),
          ("backend_marg_ms",    "marginalization", "#8c564b"),
          ("estimator_other_ms", "estimator other", "#bcbd22")]

cost = runtime_cost(TMP)
if cost.empty:
    print(f"No timing CSVs under {TMP}")
else:
    display(cost[["extraction_time_ms", "matcher_time_ms", "total_time_ms", "fe_fps",
                  "solver_time_ms", "backend_total_time_ms", "estimator_total_time_ms", "be_fps",
                  "pipeline_fps"]].sort_index().round(2))
    _plot_stacked(cost, FE_SEG, "Front-end runtime jetson_aarch (feature-tracker thread)")
    _plot_stacked(cost, BE_SEG, "Back-end runtime jetson_aarch (estimator thread)")

In [ ]:
# ── Runtime cost TABLE: per-frame median ± std [ms], split by component ──
def runtime_stats(root):
    key = ["dataset", "mode", "method"]
    ft_rows, es_rows = [], []
    for d in sorted(Path(root).glob("*_eval_*")):
        parsed = parse_eval_dir(d.name)
        if not d.is_dir() or parsed is None:
            continue
        ds, mode, _ = parsed
        for mdir in sorted(p for p in d.iterdir() if p.is_dir()):
            method = _base(mdir.name)
            for sdir in sorted(p for p in mdir.iterdir() if p.is_dir()):
                ft = sdir / f"feature_tracker_{RUN_TAG}.csv"
                es = sdir / f"estimator_metrics_{RUN_TAG}.csv"
                if ft.exists(): ft_rows.append(pd.read_csv(ft).assign(dataset=ds, mode=mode, method=method, seq=sdir.name))
                if es.exists(): es_rows.append(pd.read_csv(es).assign(dataset=ds, mode=mode, method=method, seq=sdir.name))
    if not ft_rows:
        return pd.DataFrame()
    # Front-end: per-frame component split (after warmup), then median + std per key.
    ft = pd.concat(ft_rows, ignore_index=True)
    ft["_i"] = ft.groupby(key + ["seq"]).cumcount()
    ft = ft[ft["_i"] >= RUNTIME_WARMUP].copy()
    ft["fe_extract"] = ft["extraction_time_ms"]
    ft["fe_matcher"] = ft.get("matcher_time_ms", 0.0)
    ft["fe_other"]   = (ft["total_time_ms"] - ft["fe_extract"].fillna(0) - ft["fe_matcher"].fillna(0)).clip(lower=0)
    ft["fe_total"]   = ft["total_time_ms"]
    fcomp = ["fe_extract", "fe_matcher", "fe_other", "fe_total"]
    parts = [ft.groupby(key)[fcomp].median().add_suffix("_med"),
             ft.groupby(key)[fcomp].std().add_suffix("_std")]
    # Back-end: solver ⊂ backend_total (+marg) ⊂ estimator_total.
    if es_rows:
        es = pd.concat(es_rows, ignore_index=True)
        es["_i"] = es.groupby(key + ["seq"]).cumcount()
        es = es[es["_i"] >= RUNTIME_WARMUP].copy()
        es["be_solver"] = es.get("solver_time_ms", 0.0)
        bt = es.get("backend_total_time_ms", es.get("estimator_total_time_ms"))
        es["be_marg"]   = (bt - es["be_solver"]).clip(lower=0)
        es["be_other"]  = (es.get("estimator_total_time_ms", 0.0) - es.get("backend_total_time_ms", 0.0)).clip(lower=0)
        es["be_total"]  = es.get("estimator_total_time_ms", np.nan)
        ecomp = ["be_solver", "be_marg", "be_other", "be_total"]
        parts += [es.groupby(key)[ecomp].median().add_suffix("_med"),
                  es.groupby(key)[ecomp].std().add_suffix("_std")]
    return pd.concat(parts, axis=1)

_RT_LABELS = {"fe_extract": "extract", "fe_matcher": "matcher", "fe_other": "FE other", "fe_total": "FE total",
              "be_solver": "solve",   "be_marg": "marg",       "be_other": "BE other", "be_total": "BE total"}

stats = runtime_stats(TMP)
if stats.empty:
    print(f"No timing CSVs under {TMP}")
else:
    out = pd.DataFrame(index=stats.index)
    for col, lab in _RT_LABELS.items():
        m, s = col + "_med", col + "_std"
        if m in stats.columns:
            out[lab] = [f"{mm:.1f} ± {ss:.1f}" if np.isfinite(mm) else "—"
                        for mm, ss in zip(stats[m], stats[s].fillna(0))]
    out = out.reset_index()
    out["__o"] = out["method"].map({m: i for i, m in enumerate(METHOD_ORDER)}).fillna(999)
    out["method"] = out["method"].map(lambda m: METHOD_LABELS.get(m, m))
    out = (out.sort_values(["dataset", "mode", "__o"]).drop(columns="__o")
              .set_index(["dataset", "mode", "method"]))
    print("Per-frame runtime: median ± 1σ [ms]  (warmup-trimmed; FE = feature-tracker thread, BE = estimator thread)")
    with pd.option_context("display.max_rows", None, "display.width", 240):
        display(out)
